# Translation DGMV PFTs to ecosystem zones

In [ ]:
# Libraries
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import matplotlib.pyplot as plt

In [ ]:
# Directories
dir04 = '../paper_ecosystem_carbon/output/04_dgvm_prep/'

---

In [ ]:
# Example calculations to get carbon densities per pft area of DGVMs
# PFT carbon densties per box area in tha
cdboxarea_pft1  = 10
cdboxarea_pft2  = 15
# PFT land coverfraction in % (sum of all PFT lcf == 1)
lcf_pft1 = 0.5
lcf_pft2 = 0.25
# box area in ha
boxarea = 100

# One pft
print((cdboxarea_pft1 * boxarea) / (lcf_pft1 * boxarea))
print((cdboxarea_pft1)/(lcf_pft1)) # box area not needed
# Two pfts
print(((cdboxarea_pft1 * boxarea) + (cdboxarea_pft2 * boxarea)) / 
      ((lcf_pft1 * boxarea) + (lcf_pft2 * boxarea)))
print((cdboxarea_pft1 + cdboxarea_pft2) / (lcf_pft1 + lcf_pft2)) # box area not needed

---

### Preparations

In [ ]:
# List of ecosystem types
list_etype = ['trop_ef', 'trop_df', 'temp_ebf', 'temp_dbf', 'temp_enf',
              'temp_dnf', 'glob_shrub', 'glob_grass', 'glob_tundr', 
              'glob_mount', 'glob_crop', 'glob_past']

# List of forest ecostystem types
list_etype_forest = ['trop_ef', 'trop_df', 'temp_ebf', 'temp_dbf', 'temp_enf',
                     'temp_dnf']
# List of non-forestecosystem types
list_etype_nonforest = ['glob_shrub', 'glob_grass', 'glob_tundr', 'glob_mount']

In [ ]:
# Function to translate dgvm pft arrays to ecosystem types
def prep_da_m(ds, da, da_lcf, cveg_csoil, etype, 
              m_pft0, m_pft1=None, m_pft2=None, m_pft3=None, m_pft4=None):

    """Translates dgvm pft arrays to ecosystem types"""
    
    # Define minimum pft occurance in gridcell
    if etype in list_etype_forest:
        lcf_trsh = 0.5
    if etype in list_etype_nonforest:
        lcf_trsh = 0.5
    if etype in ['glob_crop', 'glob_past']:
        lcf_trsh = 0.1
         
    # Fill nan values
    da = da.fillna(0)
    da_lcf = da_lcf.fillna(0)
    
    # Translate pfts to ecosystem types
    if m_pft0 == None:
        da_sel = da[0]
        da_lcf_sel = da_lcf[0]
    if ((m_pft0 != None) & (m_pft1 == None) & ((m_pft2 == None)) & 
        (m_pft3 == None)):
        da_sel = da.sel(PFT=m_pft0)
        da_lcf_sel = da_lcf.sel(PFT=m_pft0)
    if ((m_pft0 != None) & (m_pft1 != None) & (m_pft2 == None) & 
        (m_pft3 == None)):        
        da_sel = (da.sel(PFT=m_pft0) + da.sel(PFT=m_pft1))
        da_lcf_sel = (da_lcf.sel(PFT=m_pft0) + da_lcf.sel(PFT=m_pft1))
    if ((m_pft0 != None) & (m_pft1 != None) & (m_pft2 != None) & 
        (m_pft3 == None)):
        da_sel = (da.sel(PFT=m_pft0) + da.sel(PFT=m_pft1) + da.sel(PFT=m_pft2))
        da_lcf_sel = (da_lcf.sel(PFT=m_pft0) + da_lcf.sel(PFT=m_pft1) + 
                      da_lcf.sel(PFT=m_pft2))
    if ((m_pft0 != None) & (m_pft1 != None) & (m_pft2 != None) & 
        (m_pft3 != None)):
        da_sel = (da.sel(PFT=m_pft0) + da.sel(PFT=m_pft1) + 
                  da.sel(PFT=m_pft2) + da.sel(PFT=m_pft3))
        da_lcf_sel = (da_lcf.sel(PFT=m_pft0) + da_lcf.sel(PFT=m_pft1) + 
                      da_lcf.sel(PFT=m_pft2) + da_lcf.sel(PFT=m_pft3))
    if ((m_pft0 != None) & (m_pft1 != None) & (m_pft2 != None) & 
        (m_pft3 != None) & (m_pft4 != None)):
        da_sel = (da.sel(PFT=m_pft0) + da.sel(PFT=m_pft1) + 
                  da.sel(PFT=m_pft2) + da.sel(PFT=m_pft3) + da.sel(PFT=m_pft4))
        da_lcf_sel = (da_lcf.sel(PFT=m_pft0) + da_lcf.sel(PFT=m_pft1) + 
                      da_lcf.sel(PFT=m_pft2) + da_lcf.sel(PFT=m_pft3) +
                      da_lcf.sel(PFT=m_pft4))
    # Drop 'PFT' coordinate of da_sel and da_lcf_sel if exist
    if 'PFT' in list(da_sel.coords):
        da_sel = da_sel.drop_vars('PFT')
    if 'PFT' in list(da_lcf_sel.coords):
        da_lcf_sel = da_lcf_sel.drop_vars('PFT')
        
    # Calculate mean carbon per land area carbon
    if m_pft0 != None:        
        if cveg_csoil in ['cveg', 'csoil']:
            da_out = (da_sel / da_lcf_sel).where(da_lcf_sel > lcf_trsh)
            ds[f'{cveg_csoil}_{etype}'] = da_out 
        if cveg_csoil == 'lcf':
            ds[f'lcf_{etype}'] = da_lcf_sel
            
    if m_pft0 == None:
        ds[f'{cveg_csoil}_{etype}'] = xr.full_like(da_sel, np.nan)

---

### CABLE-POP

In [ ]:
# Get data
ds_cablepop_cveg = xr.open_dataset(
    f'{dir04}trendy_v12_r025/CABLEPOP/CABLEPOP_S2_cVegpft_r025.nc', 
    decode_times=False)
ds_cablepop_csoil = xr.open_dataset(
    f'{dir04}trendy_v12_r025/CABLEPOP/CABLEPOP_S2_cSoilpft_r025.nc', 
    decode_times=False)
ds_cablepop_lcf = xr.open_dataset(
    f'{dir04}trendy_v12_r025/CABLEPOP/CABLEPOP_S2_landCoverFrac_r025.nc', 
    decode_times=False)

# Prepare data
ds_cablepop_cveg['time'] = ds_cablepop_cveg.time + 1700
ds_cablepop_csoil['time'] = ds_cablepop_csoil.time + 1700

da_cablepop_cveg = ds_cablepop_cveg.cVegpft \
    .sel(time=slice(2010, 2020)).mean('time') * 10
da_cablepop_csoil = ds_cablepop_csoil.cSoilpft \
    .sel(time=slice(2010, 2020)).mean('time') * 10

da_cablepop_lcf = ds_cablepop_lcf.rename(time='PFT').landCoverFrac
da_cablepop_lcf['PFT'] = range(0, 10)

# 0=Evergreen Needleleaf Forest, 
# 1=Evergreen Broadleaf Forest, 
# 2=Deciduous Needleleaf Forest, 
# 3=Deciduous Broadleaf Forest, 
# 4=Shrub, 
# 5=C3 Grass, 
# 6=C4 Grass, 
# 7=Tundra, 
# 8=Barren, 
# 9=Ice'

In [ ]:
# Translate model pft output to blue pfts
ds_cablepop = xr.Dataset()

for cveg_csoil, da in zip(['cveg', 'csoil', 'lcf'], 
                          [da_cablepop_cveg, da_cablepop_csoil, da_cablepop_lcf]):
    # Land-cover-fraction file
    da_lcf = da_cablepop_lcf
    # Translate
    prep_da_m(ds_cablepop, da, da_lcf, cveg_csoil, 'trop_ef', 0, 1) # 0=Evergreen Needleleaf Forest, 1=Evergreen Broadleaf Forest
    prep_da_m(ds_cablepop, da, da_lcf, cveg_csoil, 'trop_df', 2, 3) # 2=Deciduous Needleleaf Forest, 3=Deciduous Broadleaf Forest
    prep_da_m(ds_cablepop, da, da_lcf, cveg_csoil, 'temp_ebf', 1) # 1=Evergreen Broadleaf Forest
    prep_da_m(ds_cablepop, da, da_lcf, cveg_csoil, 'temp_dbf', 3) # 3=Deciduous Broadleaf Forest
    prep_da_m(ds_cablepop, da, da_lcf, cveg_csoil, 'temp_enf', 0) # 0=Evergreen Needleleaf Forest
    prep_da_m(ds_cablepop, da, da_lcf, cveg_csoil, 'temp_dnf', 2) # 2=Deciduous Needleleaf Forest, 
    prep_da_m(ds_cablepop, da, da_lcf, cveg_csoil, 'glob_shrub', 4) # 4=Shrub, 
    prep_da_m(ds_cablepop, da, da_lcf, cveg_csoil, 'glob_grass', 5, 6) # 5=C3 Grass, 6=C4 Grass,
    prep_da_m(ds_cablepop, da, da_lcf, cveg_csoil, 'glob_tundr', 7) # 7=Tundra,  
    prep_da_m(ds_cablepop, da, da_lcf, cveg_csoil, 'glob_mount', 4, 5) # 4=Shrub, 5=C3 Grass
    prep_da_m(ds_cablepop, da, da_lcf, cveg_csoil, 'glob_crop', None) # 
    prep_da_m(ds_cablepop, da, da_lcf, cveg_csoil, 'glob_past', 5, 6) # 5=C3 Grass, 6=C4 Grass,

In [ ]:
ds_cablepop

---

### CLASSIC

In [ ]:
# Get data
ds_classic_cveg = xr.open_dataset(
    f'{dir04}trendy_v12_r025/CLASSIC/CLASSIC_S2_cVegpft_r025.nc')
ds_classic_csoil = xr.open_dataset(
    f'{dir04}trendy_v12_r025/CLASSIC/CLASSIC_S2_cSoilpft_r025.nc')
ds_classic_lcf = xr.open_dataset(
    f'{dir04}trendy_v12_r025/CLASSIC/CLASSIC_S2_landCoverFrac_r025.nc')

# Prepare data
da_classic_cveg = ds_classic_cveg.cVeg \
    .sel(time=slice('2010', '2020')).mean('time') * 10
da_classic_csoil = ds_classic_csoil.cSoil \
    .sel(time=slice('2010', '2020')).mean('time') * 10
da_classic_lcf = ds_classic_lcf.landCoverFrac \
    .sel(time=slice('2010', '2020')).mean('time')

# 1-needleleaf evergreen, 
# 2-needleleaf deciduous, 
# 3-broadleaf evergreen, 
# 4-broadleaf cold deciduous, 
# 5-broadleaf drought/dry deciduous, 
# 6-C3 crop, 
# 7-C4 crop, 
# 8-C3 grass, 
# 9-C4 grass, 
# 10-bareground (only certain variables)

In [ ]:
# Translate model pft output to blue pfts
ds_classic = xr.Dataset()

for cveg_csoil, da in zip(['cveg', 'csoil', 'lcf'], 
                          [da_classic_cveg, da_classic_csoil, da_classic_lcf]):
    # Land-cover-fraction file
    da_lcf = da_classic_lcf
    # Translate
    prep_da_m(ds_classic, da, da_lcf, cveg_csoil, 'trop_ef', 1, 3) # 1-needleleaf evergreen, 3-broadleaf evergreen
    prep_da_m(ds_classic, da, da_lcf, cveg_csoil, 'trop_df', 2, 4, 5) # 2-needleleaf deciduous, 4-broadleaf cold deciduous, 5-broadleaf drought/dry deciduous
    prep_da_m(ds_classic, da, da_lcf, cveg_csoil, 'temp_ebf', 3) # 3-broadleaf evergreen
    prep_da_m(ds_classic, da, da_lcf, cveg_csoil, 'temp_dbf', 4, 5) # 4-broadleaf cold deciduous, 5-broadleaf drought/dry deciduous
    prep_da_m(ds_classic, da, da_lcf, cveg_csoil, 'temp_enf', 1) # 1-needleleaf evergreen
    prep_da_m(ds_classic, da, da_lcf, cveg_csoil, 'temp_dnf', 2) # 2-needleleaf deciduous
    prep_da_m(ds_classic, da, da_lcf, cveg_csoil, 'glob_shrub', None) #  
    prep_da_m(ds_classic, da, da_lcf, cveg_csoil, 'glob_grass', 8, 9) # 8-C3 grass, 9-C4 grass
    prep_da_m(ds_classic, da, da_lcf, cveg_csoil, 'glob_tundr', 8) # 8-C3 grass 
    prep_da_m(ds_classic, da, da_lcf, cveg_csoil, 'glob_mount', 8) # 8-C3 grass
    prep_da_m(ds_classic, da, da_lcf, cveg_csoil, 'glob_crop', 6, 7) # 6-C3 crop, 7-C4 crop
    prep_da_m(ds_classic, da, da_lcf, cveg_csoil, 'glob_past', 8, 9) # 8-C3 grass, 9-C4 grass

In [ ]:
ds_classic

---

### IBIS

In [ ]:
# Get data
ds_ibis_cveg = xr.open_dataset(
    f'{dir04}trendy_v12_r025/IBIS/IBIS_S2_cVegpft_r025.nc',
    decode_times=False)
ds_ibis_csoil = xr.open_dataset(
    f'{dir04}trendy_v12_r025/IBIS/IBIS_S2_cSoilpft_r025.nc',
    decode_times=False)
ds_ibis_lcf = xr.open_dataset(
    f'{dir04}trendy_v12_r025/IBIS/IBIS_S2_landCoverFrac_r025.nc',
    decode_times=False)

# Prepare data
ds_ibis_cveg['time'] = (ds_ibis_cveg.time + 1700).astype('int')
ds_ibis_csoil['time'] = (ds_ibis_csoil.time + 1700).astype('int')
ds_ibis_lcf['time'] = (ds_ibis_lcf.time + 1700).astype('int')

da_ibis_cveg = ds_ibis_cveg.cVegpft \
    .sel(time=slice(2010, 2020)).mean('time') * 10
da_ibis_csoil = ds_ibis_csoil.cSoilpft \
    .sel(time=slice(2010, 2020)).mean('time') * 10
da_ibis_lcf = ds_ibis_lcf.landCoverFrac \
    .sel(time=slice(2010, 2020)).mean('time')

# 1 enf
# 2 ebf
# 3 dnf
# 4 dbf
# 5 shrub
# 6 c3grass
# 7 c4grass
# 8 c3crop
# 9 c4crop
# 10 nonveg

In [ ]:
# Translate model pft output to blue pfts
ds_ibis = xr.Dataset()

for cveg_csoil, da in zip(['cveg', 'csoil', 'lcf'], 
                          [da_ibis_cveg, da_ibis_csoil, da_ibis_lcf]):
    # Land-cover-fraction file
    da_lcf = da_ibis_lcf
    # Translate
    prep_da_m(ds_ibis, da, da_lcf, cveg_csoil, 'trop_ef', 1, 2) # 1 enf, 2 ebf
    prep_da_m(ds_ibis, da, da_lcf, cveg_csoil, 'trop_df', 3, 4) # 3 dnf, 4 dbf
    prep_da_m(ds_ibis, da, da_lcf, cveg_csoil, 'temp_ebf', 2) # 2 ebf
    prep_da_m(ds_ibis, da, da_lcf, cveg_csoil, 'temp_dbf', 4) # 4 dbf
    prep_da_m(ds_ibis, da, da_lcf, cveg_csoil, 'temp_enf', 1) # 1 enf
    prep_da_m(ds_ibis, da, da_lcf, cveg_csoil, 'temp_dnf', 3) # 3 dnf
    prep_da_m(ds_ibis, da, da_lcf, cveg_csoil, 'glob_shrub', 5) # 5 shrub  
    prep_da_m(ds_ibis, da, da_lcf, cveg_csoil, 'glob_grass', 6, 7) # 6 c3grass, 7 c4grass
    prep_da_m(ds_ibis, da, da_lcf, cveg_csoil, 'glob_tundr', 6) # 6 c3grass 
    prep_da_m(ds_ibis, da, da_lcf, cveg_csoil, 'glob_mount', 6) # 6 c3grass
    prep_da_m(ds_ibis, da, da_lcf, cveg_csoil, 'glob_crop', 8, 9) # 8 c3crop, 9 c4crop
    prep_da_m(ds_ibis, da, da_lcf, cveg_csoil, 'glob_past', 6, 7) # 6 c3grass, 7 c4grass

---

### JSBACH

In [ ]:
# Get data
ds_jsbach_cveg = xr.open_dataset(
    f'{dir04}trendy_v12_r025/JSBACH/JSBACH_S2_cVegpft_r025.nc')
ds_jsbach_csoil = xr.open_dataset(
    f'{dir04}trendy_v12_r025/JSBACH/JSBACH_S2_cSoilpft_r025.nc')
ds_jsbach_lcf = xr.open_dataset(
    f'{dir04}trendy_v12_r025/JSBACH/JSBACH_S2_landCoverFrac_r025.nc')

# Prepare data
da_jsbach_cveg = ds_jsbach_cveg.cVegpft \
    .sel(time=slice('2010', '2020')).mean('time') * 10
da_jsbach_csoil = ds_jsbach_csoil.cSoilpft \
    .sel(time=slice('2010', '2020')).mean('time') * 10
da_jsbach_lcf = ds_jsbach_lcf.landCoverFrac

da_jsbach_cveg['PFT'] = range(1, 15)
da_jsbach_csoil['PFT'] = range(1, 15)
da_jsbach_lcf['PFT'] = range(1, 15)

# PFT1:  Bare land
# PFT2:  Glacier
# PFT3:  Tropical evergreen trees
# PFT4:  Tropical deciduous trees
# PFT5:  Extra-tropical evergreen trees
# PFT6:  Extra-tropical deciduous trees
# PFT7:  Raingreen shrubs
# PFT8:  Deciduous shrubs
# PFT9:  C3 grass
# PFT10: C4 grass
# PFT11: C3 pasture
# PFT12: C4 pasture
# PFT13: C3 Crops
# PFT14: C4 Crops

In [ ]:
# Translate model pft output to blue pfts
ds_jsbach = xr.Dataset()

for cveg_csoil, da in zip(['cveg', 'csoil', 'lcf'], 
                          [da_jsbach_cveg, da_jsbach_csoil, da_jsbach_lcf]):
    # Land-cover-fraction file
    da_lcf = da_jsbach_lcf
    # Translate
    prep_da_m(ds_jsbach, da, da_lcf, cveg_csoil, 'trop_ef', 3) # PFT3:  Tropical evergreen trees
    prep_da_m(ds_jsbach, da, da_lcf, cveg_csoil, 'trop_df', 4) # PFT4:  Tropical deciduous trees
    prep_da_m(ds_jsbach, da, da_lcf, cveg_csoil, 'temp_ebf', 5) # PFT5:  Extra-tropical evergreen trees
    prep_da_m(ds_jsbach, da, da_lcf, cveg_csoil, 'temp_dbf', 6) # PFT6:  Extra-tropical deciduous trees
    prep_da_m(ds_jsbach, da, da_lcf, cveg_csoil, 'temp_enf', 5) # PFT5:  Extra-tropical evergreen trees
    prep_da_m(ds_jsbach, da, da_lcf, cveg_csoil, 'temp_dnf', 6) # PFT6:  Extra-tropical deciduous trees
    prep_da_m(ds_jsbach, da, da_lcf, cveg_csoil, 'glob_shrub', 7, 8) # PFT7:  Raingreen shrubs, PFT8:  Deciduous shrubs
    prep_da_m(ds_jsbach, da, da_lcf, cveg_csoil, 'glob_grass', 9, 10) # PFT9:  C3 grass, PFT10: C4 grass
    prep_da_m(ds_jsbach, da, da_lcf, cveg_csoil, 'glob_tundr', 7, 8, 9) # PFT7:  Raingreen shrubs, PFT8:  Deciduous shrubs, PFT9:  C3 grass   
    prep_da_m(ds_jsbach, da, da_lcf, cveg_csoil, 'glob_mount', 7, 8, 9) # PFT7:  Raingreen shrubs, PFT8:  Deciduous shrubs, PFT9:  C3 grass
    prep_da_m(ds_jsbach, da, da_lcf, cveg_csoil, 'glob_crop', 13, 14) # PFT13: C3 Crops, PFT14: C4 Crops
    prep_da_m(ds_jsbach, da, da_lcf, cveg_csoil, 'glob_past', 11, 12) # PFT11: C3 pasture, PFT12: C4 pasture

In [ ]:
ds_jsbach

---

### JULES

In [ ]:
# Get data
ds_jules_cveg = xr.open_dataset(
    f'{dir04}trendy_v12_r025/JULES/JULES_S2_cVegpft_r025.nc')
ds_jules_lcf = xr.open_dataset(
    f'{dir04}trendy_v12_r025/JULES/JULES_S2_landCoverFrac_r025.nc')

# Prepare data
da_jules_cveg = ds_jules_cveg.cVegpft \
    .rename(pft='PFT').sel(time=slice('2010', '2020')).mean('time') * 10
da_jules_lcf = ds_jules_lcf.landCoverFrac \
    .rename(tile='PFT').sel(time=slice('2010', '2020')).mean('time')

# 1 BL Decid
# 2 BL Everg - Trop
# 3 BL Everg - Temp 
# 4 NL Decid 
# 5 NL Everg
# 6 C3 Grass
# 7 C3 Crop
# 8 C3 Pasture
# 9 C4 Grass
# 10 C4 Crop
# 11 C4 Pasture
# 12 Shrub Decid
# 13 Shrub Everg

In [ ]:
# Translate model pft output to blue pfts
ds_jules = xr.Dataset()

for cveg_csoil, da in zip(['cveg', 'lcf'], [da_jules_cveg, da_jules_lcf]):
    # Land-cover-fraction file
    da_lcf = da_jules_lcf
    # Translate
    prep_da_m(ds_jules, da, da_lcf, cveg_csoil, 'trop_ef', 2, 5) # 2 BL Everg - Trop, 5 NL Everg
    prep_da_m(ds_jules, da, da_lcf, cveg_csoil, 'trop_df', 1, 4) # 1 BL Decid - 4 NL Decid 
    prep_da_m(ds_jules, da, da_lcf, cveg_csoil, 'temp_ebf', 3) # 3 BL Everg - Temp 
    prep_da_m(ds_jules, da, da_lcf, cveg_csoil, 'temp_dbf', 1) # 1 BL Decid
    prep_da_m(ds_jules, da, da_lcf, cveg_csoil, 'temp_enf', 5) # 5 NL Everg
    prep_da_m(ds_jules, da, da_lcf, cveg_csoil, 'temp_dnf', 4) # 4 NL Decid
    prep_da_m(ds_jules, da, da_lcf, cveg_csoil, 'glob_shrub', 12, 13) # 12 Shrub Decid, 13 Shrub Everg
    prep_da_m(ds_jules, da, da_lcf, cveg_csoil, 'glob_grass', 6, 9) # 6 C3 Grass, 9 C4 Grass
    prep_da_m(ds_jules, da, da_lcf, cveg_csoil, 'glob_tundr', 6, 12, 13) # 6 C3 Grass, 12 Shrub Decid, 13 Shrub Everg   
    prep_da_m(ds_jules, da, da_lcf, cveg_csoil, 'glob_mount', 6, 12 , 13) # 6 C3 Grass, 12 Shrub Decid, 13 Shrub Everg
    prep_da_m(ds_jules, da, da_lcf, cveg_csoil, 'glob_crop', 7, 10) # 7 C3 Crop, 10 C4 Crop
    prep_da_m(ds_jules, da, da_lcf, cveg_csoil, 'glob_past', 8, 11) # 8 C3 Pasture, 11 C4 Pasture

In [ ]:
ds_jules

---

### LPJ-GUESS

In [ ]:
# Get data
ds_lpjguess_cveg = xr.open_dataset(
    f'{dir04}trendy_v12_r025/LPJ-GUESS/LPJ-GUESS_S2_cVegpft_r025.nc')
ds_lpjguess_lcf = xr.open_dataset(
    f'{dir04}trendy_v12_r025/LPJ-GUESS/LPJ-GUESS_S2_landCoverFrac_r025.nc')

# Prepare data
da_lpjguess_cveg = ds_lpjguess_cveg.cVegpft \
    .sel(time=slice('2010', '2020')).mean('time') \
    .transpose('PFT', 'lat', 'lon') * 10
da_lpjguess_lcf = ds_lpjguess_lcf.landCoverFrac \
    .sel(time=slice('2010', '2020')).mean('time') \
    .transpose('PFT', 'lat', 'lon')

# 1:Boreal needleleaf evergreen (BNE), 
# 2:Boreal shade-intolerant needleleaf evergreen (BINE), 
# 3:Boreal needleleaf summergreen (BNS), 
# 4:Temperate needleleaf evergreen (TeNE), 
# 5:Temperate broadleaf summergreen (TeBS), 
# 6:Temperate shade-intolerant broadleaf summergreen (IBS), 
# 7:Temperate broadleaf evergreen (TeBE), 
# 8:Tropical broadleaf evergreen (TrBE), 
# 9:Tropical shade-intolerant broadleaf evergreen (TrIBE), 
# 10:Tropical broadleaf raingreen (TrBR), 
# 11:C3 grass (C3G), 
# 12:C4 grass (C4G), 
# 13:C3 grass in pasture (C3G_pas), 
# 14:C4 grass in pasture (C4G_pas), 
# 15:C3 annual crops (wheat w/o ccg) (CC3ann), 
# 16:C3 perennial crops (summer wheat w/ ccg) (CC3per), 
# 17:C3 nitrogen-fixing crops (summer wheat w/o ccg) (CC3nfx), 
# 18:C4 annual crops (corn w/o ccg) (CC4ann), 
# 19:C4 perennial crops (corn w/ ccg) (CC4per), 
# 20:C3 annual crops irrigated (wheat w/o ccg) (CC3anni), 
# 21:C3 perennial crops irrigated (summer wheat w/ ccg) (CC3peri), 
# 22:C3 nitrogen-fixing crops irrigated (summer wheat w/o ccg) (CC3nfxi), 
# 23:C4 annual crops irrigated (corn w/o ccg) (CC4anni), 
# 24:C4 perennial crops irrigated (corn w/ ccg) (CC4peri), 
# 25:C3 cover crop grass (ccg) (CC3G_ic), 
# 26:C4 cover crop grass (ccg) (CC4G_ic)

In [ ]:
# Translate model pft output to blue pfts
ds_lpjguess = xr.Dataset()

for cveg_csoil, da in zip(['cveg', 'lcf'], [da_lpjguess_cveg, da_lpjguess_lcf]):
    # Land-cover-fraction file
    da_lcf = da_lpjguess_lcf
    # Translate
    prep_da_m(ds_lpjguess, da, da_lcf, cveg_csoil, 'trop_ef', 8, 9) # 8:Tropical broadleaf evergreen (TrBE), 9:Tropical shade-intolerant broadleaf evergreen (TrIBE)
    prep_da_m(ds_lpjguess, da, da_lcf, cveg_csoil, 'trop_df', 10) # 10:Tropical broadleaf raingreen (TrBR)
    prep_da_m(ds_lpjguess, da, da_lcf, cveg_csoil, 'temp_ebf', 7) # 7:Temperate broadleaf evergreen (TeBE)
    prep_da_m(ds_lpjguess, da, da_lcf, cveg_csoil, 'temp_dbf', 5, 6) # 5:Temperate broadleaf summergreen (TeBS), 6:Temperate shade-intolerant broadleaf summergreen (IBS)
    prep_da_m(ds_lpjguess, da, da_lcf, cveg_csoil, 'temp_enf', 1, 2, 4) # 1:Boreal needleleaf evergreen (BNE), 2:Boreal shade-intolerant needleleaf evergreen (BINE), 4:Temperate needleleaf evergreen (TeNE)
    prep_da_m(ds_lpjguess, da, da_lcf, cveg_csoil, 'temp_dnf', 3) # 3:Boreal needleleaf summergreen (BNS)
    prep_da_m(ds_lpjguess, da, da_lcf, cveg_csoil, 'glob_shrub', None) 
    prep_da_m(ds_lpjguess, da, da_lcf, cveg_csoil, 'glob_grass', 11, 12) # 11:C3 grass (C3G), 12:C4 grass (C4G)
    prep_da_m(ds_lpjguess, da, da_lcf, cveg_csoil, 'glob_tundr', 11) # 11:C3 grass (C3G) 
    prep_da_m(ds_lpjguess, da, da_lcf, cveg_csoil, 'glob_mount', 11) # 11:C3 grass (C3G)
    prep_da_m(ds_lpjguess, da, da_lcf, cveg_csoil, 'glob_crop', 15, 16, 17, 18, 19) # 15:C3 annual crops (wheat w/o ccg) (CC3ann), 16:C3 perennial crops (summer wheat w/ ccg) (CC3per), 17:C3 nitrogen-fixing crops (summer wheat w/o ccg) (CC3nfx), 18:C4 annual crops (corn w/o ccg) (CC4ann), 19:C4 perennial crops (corn w/ ccg) (CC4per), 
    prep_da_m(ds_lpjguess, da, da_lcf, cveg_csoil, 'glob_past', 13, 14) # 13:C3 grass in pasture (C3G_pas), 14:C4 grass in pasture (C4G_pas)

In [ ]:
ds_lpjguess

---

### LPJmL

In [ ]:
# Note: cVegpft file has different number of PFTs than landCoverFrac file

# Get data
ds_lpjml_cveg = xr.open_dataset(
    f'{dir04}trendy_v12_r025/LPJmL/LPJmL_S2_cVegpft_r025.nc', 
    decode_times=False)
ds_lpjml_lcf = xr.open_dataset(
    f'{dir04}trendy_v12_r025/LPJmL/LPJmL_S2_landCoverFrac_r025.nc', 
    decode_times=False)

# Prepare data
ds_lpjml_cveg['time'] = (ds_lpjml_cveg.time + 1700).astype('int')
ds_lpjml_lcf['time'] = (ds_lpjml_lcf.time + 1700).astype('int')

da_lpjml_cveg = ds_lpjml_cveg.cVegpft \
    .sel(time=slice(2010, 2020)).mean('time') * 10
da_lpjml_lcf = ds_lpjml_lcf.landCoverFrac \
    .sel(time=slice(2010, 2020)).mean('time')

#cVeg
#0 b'tropical broadleaved evergreen tree',
#1 b'tropical broadleaved raingreen tree',
#2 b'temperate needleleaved evergreen tree',
#3 b'temperate broadleaved evergreen tree',
#4 b'temperate broadleaved summergreen tree',
#5 b'boreal needleleaved evergreen tree',
#6 b'boreal broadleaved summergreen tree',
#7 b'boreal needleleaved summergreen tree', 
#8 b'Tropical C4 grass',
#9 b'Temperate C3 grass', 
#10 b'Polar C3 grass', 
#11 b'cropland',
#12 b'permanent pastures'

#landCoverFrac
#0 b'tropical broadleaved evergreen tree',
#1 b'tropical broadleaved raingreen tree',
#2 b'temperate needleleaved evergreen tree',
#3 b'temperate broadleaved evergreen tree',
#4 b'temperate broadleaved summergreen tree',
#5 b'boreal needleleaved evergreen tree',
#6 b'boreal broadleaved summergreen tree',
#7 b'boreal needleleaved summergreen tree', 
#8 b'Tropical C4 grass',
#9 b'Temperate C3 grass', 
#10 b'Polar C3 grass',
#11 b'natural vegetation bare soil', 
#12 b'cropland', 
#13 b'permanent pastures',
#14 b'rivers, lakes, and reservoirs'

In [ ]:
# Translate model pft output to blue pfts
ds_lpjml = xr.Dataset()

for cveg_csoil, da in zip(['cveg', 'lcf'], [da_lpjml_cveg, da_lpjml_lcf]):
    # Land-cover-fraction file
    da_lcf = da_lpjml_lcf
    # Translate
    prep_da_m(ds_lpjml, da, da_lcf, cveg_csoil, 'trop_ef', 0) # 0 b'tropical broadleaved evergreen tree'
    prep_da_m(ds_lpjml, da, da_lcf, cveg_csoil, 'trop_df', 1) # 1 b'tropical broadleaved raingreen tree'
    prep_da_m(ds_lpjml, da, da_lcf, cveg_csoil, 'temp_ebf', 3) # 3 b'temperate broadleaved evergreen tree'
    prep_da_m(ds_lpjml, da, da_lcf, cveg_csoil, 'temp_dbf', 4, 6) # 4 b'temperate broadleaved summergreen tree', 6 b'boreal broadleaved summergreen tree'
    prep_da_m(ds_lpjml, da, da_lcf, cveg_csoil, 'temp_enf', 2, 5) # 2 b'temperate needleleaved evergreen tree', 5 b'boreal needleleaved evergreen tree'
    prep_da_m(ds_lpjml, da, da_lcf, cveg_csoil, 'temp_dnf', 7) # 7 b'boreal needleleaved summergreen tree' 
    prep_da_m(ds_lpjml, da, da_lcf, cveg_csoil, 'glob_shrub', None) # 
    prep_da_m(ds_lpjml, da, da_lcf, cveg_csoil, 'glob_grass', 8, 9, 10) # 8 b'Tropical C4 grass', 9 b'Temperate C3 grass', 10 b'Polar C3 grass'
    prep_da_m(ds_lpjml, da, da_lcf, cveg_csoil, 'glob_tundr', 9, 10) # 9 b'Temperate C3 grass', 10 b'Polar C3 grass' 
    prep_da_m(ds_lpjml, da, da_lcf, cveg_csoil, 'glob_mount', 9, 10) # 9 b'Temperate C3 grass', 10 b'Polar C3 grass'
    prep_da_m(ds_lpjml, da, da_lcf, cveg_csoil, 'glob_crop', None) # numbers of lcf and cveg pfts do not fit
    prep_da_m(ds_lpjml, da, da_lcf, cveg_csoil, 'glob_past', None) # numbers of lcf and cveg pfts do not fit 

In [ ]:
ds_lpjml

---

### OCN

In [ ]:
# Get data
ds_ocn_cveg = xr.open_dataset(
    f'{dir04}trendy_v12_r025/OCN/OCN_S2_cVegpft_r025.nc')
ds_ocn_csoil = xr.open_dataset(
    f'{dir04}trendy_v12_r025/OCN/OCN_S2_cSoilpft_r025.nc')
ds_ocn_lcf = xr.open_dataset(
    f'{dir04}trendy_v12_r025/OCN/OCN_S2_landCoverFrac_r025.nc')

# Prepare data
da_ocn_cveg = ds_ocn_cveg.cVegpft \
    .sel(time=slice('2010', '2020')).mean('time') * 0.01
da_ocn_csoil = ds_ocn_csoil.cSoilpft \
    .sel(time=slice('2010', '2020')).mean('time') * 0.01
da_ocn_lcf = ds_ocn_lcf.landCoverFrac \
     .sel(time=slice('2010', '2020')).mean('time')

# PFT1: Bare soil 
# PFT2: TrBE - Tropical Broadleaf Evergreen
# PFT3: TrBR - Tropical Broadleaf Raingreen
# PFT4: TeNE - Temperate Needleleaf Evergreen
# PFT5: TeBE - Temperate Broadleaf Evergreen
# PFT6: TeBS - Temperate Broadleaf Summergreen
# PFT7: BNE - Boreal Needleleaf Evergreen
# PFT8: BBS - Boreal Broadleaf Summergreen
# PFT9: BNS - Boreal Needleleaf Summergreen
# PFT10: TeH - Temperate Herbaceous
# PFT11: TrH - Tropical Herbaceous
# PFT12: TeH_crop - Temperate Herbaceous Crop
# PFT13: TrH_crop - Tropical Herbaceous Crop

In [ ]:
# Translate model pft output to blue pfts
ds_ocn = xr.Dataset()

for cveg_csoil, da in zip(['cveg', 'csoil', 'lcf'], 
                          [da_ocn_cveg, da_ocn_csoil, 
                           da_ocn_lcf]):
    # Land-cover-fraction file
    da_lcf = da_ocn_lcf
    # Translate
    prep_da_m(ds_ocn, da, da_lcf, cveg_csoil, 'trop_ef', 2) # PFT2: TrBE - Tropical Broadleaf Evergreen 
    prep_da_m(ds_ocn, da, da_lcf, cveg_csoil, 'trop_df', 3) # PFT3: TrBR - Tropical Broadleaf Raingreen (or Deciduous) 
    prep_da_m(ds_ocn, da, da_lcf, cveg_csoil, 'temp_ebf', 5) # PFT5: TeBE - Temperate Broadleaf Evergreen, 
    prep_da_m(ds_ocn, da, da_lcf, cveg_csoil, 'temp_dbf', 6, 8) # PFT6: TeBS - Temperate Broadleaf Summergreen (or Deciduous), PFT8: BBS - Boreal Broadleaf Summergreen (or Deciduous)
    prep_da_m(ds_ocn, da, da_lcf, cveg_csoil, 'temp_enf', 4, 7) # PFT4: TeNE - Temperate Needleleaf Evergreen, PFT7: BNE - Boreal Needleleaf Evergreen
    prep_da_m(ds_ocn, da, da_lcf, cveg_csoil, 'temp_dnf', 9) # PFT9: BNS - Boreal Needleleaf Summergreen (or Deciduous)
    prep_da_m(ds_ocn, da, da_lcf, cveg_csoil, 'glob_shrub', 10, 11) # PFT10: TeH - Temperate Herbaceous, PFT11: TrH - Tropical Herbaceous
    prep_da_m(ds_ocn, da, da_lcf, cveg_csoil, 'glob_grass', None) # 
    prep_da_m(ds_ocn, da, da_lcf, cveg_csoil, 'glob_tundr', None) # 
    prep_da_m(ds_ocn, da, da_lcf, cveg_csoil, 'glob_mount', None) # 
    prep_da_m(ds_ocn, da, da_lcf, cveg_csoil, 'glob_crop', None) # 12 and 13 have much higher values than other DGVMs
    prep_da_m(ds_ocn, da, da_lcf, cveg_csoil, 'glob_past', None) # 

In [ ]:
ds_ocn

---

### ORCHIDEE

In [ ]:
# Get data
ds_orchidee_cveg = xr.open_dataset(
    f'{dir04}trendy_v12_r025/ORCHIDEE/ORCHIDEE_S2_cVegpft_r025.nc')
ds_orchidee_csoil = xr.open_dataset(
    f'{dir04}trendy_v12_r025/ORCHIDEE/ORCHIDEE_S2_cSoilpft_r025.nc')
ds_orchidee_lcf = xr.open_dataset(
    f'{dir04}trendy_v12_r025/ORCHIDEE/ORCHIDEE_S2_landCoverFrac_r025.nc')

# Prepare data
da_orchidee_cveg = ds_orchidee_cveg.cVegpft \
    .rename(pft='PFT').sel(time=slice('2010', '2020')).mean('time') * 10
da_orchidee_csoil = ds_orchidee_csoil.cSoilpft \
    .rename(pft='PFT').sel(time=slice('2010', '2020')).mean('time') * 10
da_orchidee_lcf = ds_orchidee_lcf.landCoverFrac \
    .rename(pft='PFT').sel(time=slice('2010', '2020')).mean('time')

# 0 - 'PFT1: bare soil',
# 1 - 'PFT2: tropical broadleaf evergreen',
# 2 - 'PFT3: tropical broadleaf raingreen',
# 3 - 'PFT4: temperate needleleaf evergreen',
# 4 - 'PFT5: temperate broadleaf evergreen',
# 5 - 'PFT6: temperate broadleaf summergreen',
# 6 - 'PFT7: boreal needleleaf evergreen',
# 7 - 'PFT8: boreal broadleaf summergreen',
# 8 - 'PFT9: boreal needleleaf summergreen',
# 9 - 'PFT10: temperate C3 grass',
# 10 - 'PFT11: C4 grass',
# 11 - 'PFT12: C3 agriculture',
# 12 - 'PFT13: C4 agriculture',
# 13 - 'PFT14: tropical C3 grass',
# 14 - 'PFT15: boreal C3 grass'

In [ ]:
# Translate model pft output to blue pfts
ds_orchidee = xr.Dataset()

for cveg_csoil, da in zip(['cveg', 'csoil', 'lcf'], 
                          [da_orchidee_cveg, da_orchidee_csoil, 
                           da_orchidee_lcf]):
    # Land-cover-fraction file
    da_lcf = da_orchidee_lcf
    # Translate
    prep_da_m(ds_orchidee, da, da_lcf, cveg_csoil, 'trop_ef', 1) # '1 - 'PFT2: tropical broadleaf evergreen',
    prep_da_m(ds_orchidee, da, da_lcf, cveg_csoil, 'trop_df', 2) # 2 - 'PFT3: tropical broadleaf raingreen',
    prep_da_m(ds_orchidee, da, da_lcf, cveg_csoil, 'temp_ebf', 4) # 4 - 'PFT5: temperate broadleaf evergreen',
    prep_da_m(ds_orchidee, da, da_lcf, cveg_csoil, 'temp_dbf', 5, 7) # 5 - 'PFT6: temperate broadleaf summergreen', 7 - 'PFT8: boreal broadleaf summergreen',
    prep_da_m(ds_orchidee, da, da_lcf, cveg_csoil, 'temp_enf', 3, 6) # 3 - 'PFT4: temperate needleleaf evergreen', 5 - 'PFT6: temperate broadleaf summergreen',
    prep_da_m(ds_orchidee, da, da_lcf, cveg_csoil, 'temp_dnf', 8) # 8 - 'PFT9: boreal needleleaf summergreen',
    prep_da_m(ds_orchidee, da, da_lcf, cveg_csoil, 'glob_shrub', None) # 
    prep_da_m(ds_orchidee, da, da_lcf, cveg_csoil, 'glob_grass', 9, 10, 13, 14) # 9 - 'PFT10: temperate C3 grass', 10 - 'PFT11: C4 grass', 13 - 'PFT14: tropical C3 grass', 14 - 'PFT15: boreal C3 grass'
    prep_da_m(ds_orchidee, da, da_lcf, cveg_csoil, 'glob_tundr', 14) # 14 - 'PFT15: boreal C3 grass'
    prep_da_m(ds_orchidee, da, da_lcf, cveg_csoil, 'glob_mount', 9, 13, 14) # 9 - 'PFT10: temperate C3 grass', 13 - 'PFT14: tropical C3 grass', 14 - 'PFT15: boreal C3 grass'
    prep_da_m(ds_orchidee, da, da_lcf, cveg_csoil, 'glob_crop', 11, 12) # 11 - 'PFT12: C3 agriculture', 12 - 'PFT13: C4 agriculture',
    prep_da_m(ds_orchidee, da, da_lcf, cveg_csoil, 'glob_past', 9, 10, 13, 14) # 9 - 'PFT10: temperate C3 grass', 10 - 'PFT11: C4 grass', 13 - 'PFT14: tropical C3 grass', 14 - 'PFT15: boreal C3 grass'

In [ ]:
ds_orchidee

---

### SDGVM

In [ ]:
# Get data
ds_sdgvm_cveg = xr.open_dataset(dir04 + 'trendy_v12_r025/SDGVM/SDGVM_S2_cVegpft_r025.nc')
ds_sdgvm_lcf = xr.open_dataset(dir04 + 'trendy_v12_r025/SDGVM/SDGVM_S2_landCoverFrac_r025.nc')

# Prepare data
da_sdgvm_cveg = ds_sdgvm_cveg.cVegpft \
    .sel(time=slice('2010', '2020')).mean('time') * 10
da_sdgvm_lcf = ds_sdgvm_lcf.landCoverFrac \
    .sel(time=slice('2010', '2020')).mean('time')

# 1 BARE: Bare ground.
# 2 CITY: Urban areas.
# 3 C3p: C3 perennial grasses.
# 4 C3s: C3 short-lived grasses.
# 5 C3crop: C3 crops.
# 6 C4p: C4 perennial grasses.
# 7 C4s: C4 short-lived grasses.
# 8 C4crop: C4 crops.
# 9 Dc_Bp: Deciduous broadleaf perennial trees.
# 10 Dc_Np: Deciduous needleleaf perennial trees.
# 11 Ev_Bp: Evergreen broadleaf perennial trees.
# 12 Ev_Np: Evergreen needleleaf perennial trees.
# 13 Dc_Bs: Deciduous broadleaf shrubs.
# 14 Dc_Ns: Deciduous needleleaf shrubs.
# 15 Ev_Bs: Evergreen broadleaf shrubs.
# 16 Ev_Ns: Evergreen needleleaf shrubs.

In [ ]:
# Translate model pft output to blue pfts
ds_sdgvm = xr.Dataset()

for cveg_csoil, da in zip(['cveg', 'lcf'], [da_sdgvm_cveg, da_sdgvm_lcf]):
    # Land-cover-fraction file
    da_lcf = da_sdgvm_lcf
    # Translate
    prep_da_m(ds_sdgvm, da, da_lcf, cveg_csoil, 'trop_ef', 11, 12) # 11 Ev_Bp, 12 Ev_Np
    prep_da_m(ds_sdgvm, da, da_lcf, cveg_csoil, 'trop_df', 9, 10) # 9 Dc_Bp, 10 Dc_Np
    prep_da_m(ds_sdgvm, da, da_lcf, cveg_csoil, 'temp_ebf', 11) # 11 Ev_Bp
    prep_da_m(ds_sdgvm, da, da_lcf, cveg_csoil, 'temp_dbf', 9) # 9 Dc_Bp  
    prep_da_m(ds_sdgvm, da, da_lcf, cveg_csoil, 'temp_enf', 12) # 12 Ev_Np
    prep_da_m(ds_sdgvm, da, da_lcf, cveg_csoil, 'temp_dnf', 10) # 10 Dc_Np  
    prep_da_m(ds_sdgvm, da, da_lcf, cveg_csoil, 'glob_shrub', 13, 14, 15, 16) # 13 Dc_Bs, 14 Dc_Ns, 15 Ev_Bs, 16 Ev_Ns  
    prep_da_m(ds_sdgvm, da, da_lcf, cveg_csoil, 'glob_grass', 3, 4, 6, 7) # 3 C3p, 4 C3s, 6 C4p, 7 C4s
    prep_da_m(ds_sdgvm, da, da_lcf, cveg_csoil, 'glob_tundr', 3, 4) # # 3 C3p, 4 C3s   
    prep_da_m(ds_sdgvm, da, da_lcf, cveg_csoil, 'glob_mount', 3, 4) # # 3 C3p, 4 C3s
    prep_da_m(ds_sdgvm, da, da_lcf, cveg_csoil, 'glob_crop', 5, 8) # 5 C3crop, 8 C4crop
    prep_da_m(ds_sdgvm, da, da_lcf, cveg_csoil, 'glob_past', None) # 

In [ ]:
ds_sdgvm

---

### YIBS

In [ ]:
# Get data
ds_yibs_cveg = xr.open_dataset(
    f'{dir04}trendy_v12_r025/YIBS/YIBS_S2_cVegpft_r025.nc',
    decode_times=False)
ds_yibs_csoil = xr.open_dataset(
    f'{dir04}trendy_v12_r025/YIBS/YIBS_S2_cSoilpft_r025.nc',
    decode_times=False)
ds_yibs_lcf = xr.open_dataset(
    f'{dir04}trendy_v12_r025/YIBS/YIBS_S2_landCoverFrac_r025.nc',
    decode_times=False)

# Prepare data
ds_yibs_cveg['time'] = (ds_yibs_cveg.time + 1700).astype('int')
ds_yibs_csoil['time'] = (ds_yibs_csoil.time + 1700).astype('int')
ds_yibs_lcf['time'] = (ds_yibs_lcf.time + 1700).astype('int')

da_yibs_cveg = ds_yibs_cveg.cVegpft \
    .rename(pfts='PFT').sel(time=slice(2010, 2020)).mean('time') * 10
da_yibs_csoil = ds_yibs_csoil.cSoilpft \
    .rename(pfts='PFT').sel(time=slice(2010, 2020)).mean('time') * 10
da_yibs_lcf = ds_yibs_lcf.landCoverFrac \
    .rename(pfts='PFT').sel(time=slice(2010, 2020)).mean('time')

# 1 'PFT_01': 'Evergreen Broadleaf Forest',
# 2 'PFT_02': 'Evergreen Needleleaf Forest',
# 3 'PFT_03': 'Deciduous Broadleaf Forest',
# 4 'PFT_04': 'Shrubland',
# 5 'PFT_05': 'C3 grassland',
# 6 'PFT_06': 'C4 grassland',
# 7 'PFT_07': 'C3 cropland',
# 8 'PFT_08': 'C4 cropland'

In [ ]:
# Translate model pft output to blue pfts
ds_yibs = xr.Dataset()

for cveg_csoil, da in zip(['cveg', 'csoil', 'lcf'], 
                          [da_yibs_cveg, da_yibs_csoil, da_yibs_lcf]):
    # Land-cover-fraction file
    da_lcf = da_yibs_lcf
    # Translate
    prep_da_m(ds_yibs, da, da_lcf, cveg_csoil, 'trop_ef', 1, 2) #  1 'PFT_01': 'Evergreen Broadleaf Forest', 2 'PFT_02': 'Evergreen Needleleaf Forest',
    prep_da_m(ds_yibs, da, da_lcf, cveg_csoil, 'trop_df', 3) #  3 'PFT_03': 'Deciduous Broadleaf Forest',
    prep_da_m(ds_yibs, da, da_lcf, cveg_csoil, 'temp_ebf', 1) #  1 'PFT_01': 'Evergreen Broadleaf Forest',
    prep_da_m(ds_yibs, da, da_lcf, cveg_csoil, 'temp_dbf', 3) #  3 'PFT_03': 'Deciduous Broadleaf Forest',
    prep_da_m(ds_yibs, da, da_lcf, cveg_csoil, 'temp_enf', 2) # 2 'PFT_02': 'Evergreen Needleleaf Forest',
    prep_da_m(ds_yibs, da, da_lcf, cveg_csoil, 'temp_dnf', None) # 
    prep_da_m(ds_yibs, da, da_lcf, cveg_csoil, 'glob_shrub', 4) # 4 'PFT_04': 'Shrubland',
    prep_da_m(ds_yibs, da, da_lcf, cveg_csoil, 'glob_grass', 5, 6) # 5 'PFT_05': 'C3 grassland', 6 'PFT_06': 'C4 grassland',
    prep_da_m(ds_yibs, da, da_lcf, cveg_csoil, 'glob_tundr', 4, 5) # 4 'PFT_04': 'Shrubland', 5 'PFT_05': 'C3 grassland'  
    prep_da_m(ds_yibs, da, da_lcf, cveg_csoil, 'glob_mount', 4, 5) # 4 'PFT_04': 'Shrubland', 5 'PFT_05': 'C3 grassland'
    prep_da_m(ds_yibs, da, da_lcf, cveg_csoil, 'glob_crop', 7, 8) # 7 'PFT_07': 'C3 cropland', 8 'PFT_08': 'C4 cropland'
    prep_da_m(ds_yibs, da, da_lcf, cveg_csoil, 'glob_past', 5, 6) # 5 'PFT_05': 'C3 grassland', 6 'PFT_06': 'C4 grassland'

In [ ]:
ds_yibs

---

### Merge datasets

In [ ]:
def prep_dgmv_list(ctype):
    
    if ctype in ['cveg', 'lcf']:
        return [ds_cablepop[f'{ctype}_{etype}'].rename('cablepop'),
                ds_classic[f'{ctype}_{etype}'].rename('classic'),
                ds_ibis[f'{ctype}_{etype}'].rename('ibis'),
                ds_jsbach[f'{ctype}_{etype}'].rename('jsbach'),
                ds_jules[f'{ctype}_{etype}'].rename('jules'),
                ds_lpjguess[f'{ctype}_{etype}'].rename('lpjguess'),
                ds_lpjml[f'{ctype}_{etype}'].rename('lpjml'),
                ds_ocn[f'{ctype}_{etype}'].rename('ocn'),
                ds_orchidee[f'{ctype}_{etype}'].rename('orchidee'),
                ds_sdgvm[f'{ctype}_{etype}'].rename('sdgvm'),
                ds_yibs[f'{ctype}_{etype}'].rename('yibs')]
        
    if ctype == 'csoil':
        return [ds_cablepop[f'{ctype}_{etype}'].rename('cablepop'),
                ds_classic[f'{ctype}_{etype}'].rename('classic'),
                ds_ibis[f'{ctype}_{etype}'].rename('ibis'),
                ds_jsbach[f'{ctype}_{etype}'].rename('jsbach'),
                ds_ocn[f'{ctype}_{etype}'].rename('ocn'),
                ds_orchidee[f'{ctype}_{etype}'].rename('orchidee'),
                ds_yibs[f'{ctype}_{etype}'].rename('yibs')]

ds_dgvm = xr.Dataset()

for etype in list_etype:
    for ctype in ['cveg', 'csoil', 'lcf']:
        ds_dgvm[f'{ctype}_{etype}'] = xr.merge(prep_dgmv_list(ctype)) \
            .to_array('model')

        
#ds_dgvm.to_netcdf(dir04 + 'ds_dgvm_translated_r025.nc', mode='w')
ds_dgvm.to_zarr(dir04 + 'ds_dgvm_translated_r025.zarr', mode='w');